# Redes Convolucionales con PyTorch

Usaremos CIFAR-10. https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.CIFAR10.html#torchvision.datasets.CIFAR10

In [1]:
import random

import torch.nn as nn
import pandas as pd
import torch
import torchvision
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()

True

### Load data

In [ ]:
IMG_HEIGHT = 32
IMG_WIDTH = 32
IMG_CHANNELS = 3
batch_size = 32

# Me basé en https://www.geeksforgeeks.org/python/how-to-load-cifar10-dataset-in-pytorch/
# Nota: TV = TorchVision
train_as_seen_in_tv = torchvision.datasets.CIFAR10(root='./datasets/cifar10', train=True, download=True, transform=torchvision.transforms.Compose([torchvision.transforms.ToTensor(),torchvision.transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),torchvision.transforms.Resize((IMG_HEIGHT, IMG_WIDTH))]))
test_as_seen_in_tv = torchvision.datasets.CIFAR10(root='./datasets/cifar10', train=False, download=True, transform=torchvision.transforms.Compose([torchvision.transforms.ToTensor(),torchvision.transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),torchvision.transforms.Resize((IMG_HEIGHT, IMG_WIDTH))]))

train_loader_from_tv_at_ONLY_32_batch = DataLoader(train_as_seen_in_tv, batch_size=batch_size, shuffle=True)
test_loader_from_tv_at_ONLY_32_batch = DataLoader(test_as_seen_in_tv, batch_size=batch_size, shuffle=False)

/home/david/computational-intelligence/venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


## Preparing dataset

## Training functions

Now we can define our training functions. Since we will use accuracy to evaluate our model, we will create a function to calculate accuracy as well. 

In [ ]:
def get_batch_accuracy(output, y):
    pred = output.argmax(dim=1)
    correct = (pred == y).sum().item() # no me haga mucho caso pero creo que esta función estaba mal porque N es el tamaño del dataset completo y esto es un batch, incluso acertando las 32 [sic] instancias nos daría un número irremediablemente menor a 100%
    return correct/y.size(0)

Our train function performs the full training loop. For each epoch, it computes the training loss and accuracy, and then evaluates the model on the validation set. It returns the history of training and validation losses and accuracies for plotting later.

In [ ]:
def train(_model, _train_loader, _test_loader, _criterion, _optimizer, _num_epochs):
    res = {
        'train_loss': [],
        'train_acc': [],
        'test_loss': [],
        'test_acc': []
    }
    iterator = tqdm(range(_num_epochs), desc="Training", unit="epoch")

    for _ in iterator:
        _model.train()
        train_loss = 0.0
        train_acc = 0.0
        test_loss = 0.0
        test_acc = 0.0
        for X_batch, y_batch in _train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            _optimizer.zero_grad()
            outputs = _model(X_batch)
            loss = _criterion(outputs, y_batch)
            loss.backward()
            _optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
            train_acc += get_batch_accuracy(outputs, y_batch)

        epoch_train_loss = train_loss / len(_train_loader.dataset)
        
        _model.eval()

        with torch.no_grad():
            for x, y in _test_loader:
                x = x.to(device)
                y = y.to(device)

                output = _model(x)
                loss = _criterion(output, y)

                test_loss += loss.item() * x.size(0)
                test_acc += get_batch_accuracy(outputs, y)

        epoch_test_loss = test_loss / len(_test_loader.dataset)

        iterator.set_postfix(
            train_loss=f"{epoch_train_loss:.4f}",
            train_acc=f"{train_acc:.4f}",
            test_loss=f"{epoch_test_loss:.4f}",
            test_acc=f"{test_acc:.4f}"
        )

        res['train_loss'].append(epoch_train_loss)
        res['train_acc'].append(train_acc)
        res['test_loss'].append(epoch_test_loss)
        res['test_acc'].append(test_acc)

    return res

This test function will evaluate the model on the test set after training is complete. It will calculate the average loss and accuracy across the entire test set.

In [ ]:
def test(_model, _test_loader, _loss_function):
    _model.eval()
    test_loss = 0.0
    test_acc = 0.0
    with torch.no_grad():
        for x, y in _test_loader:
            x = x.to(device)
            y = y.to(device)
            output = _model(x)
            test_loss += _loss_function(output, y).item() * x.size(0)
            test_acc += get_batch_accuracy(output, y)

    return (test_loss / len(_test_loader.dataset)), test_acc

## Training the models

### CNN model

Now let's implement a simple CNN architecture. We will use three convolutional layers followed by max pooling, and then a couple of fully connected layers before the output layer.

In [7]:
model_cnn = nn.Sequential(
    nn.Conv2d(IMG_CHANNELS, 64, 3, stride=2, padding=1),
    # x: ¿por qué 64 filtros?
    # homero: para más poder
    nn.BatchNorm2d(64), # esto debiera permitirnos un aprendizaje más fino: https://www.geeksforgeeks.org/deep-learning/what-is-batch-normalization-in-deep-learning/
    nn.ReLU(),
    nn.Conv2d(64, 64, 3, padding=1),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(64, 128, 3, padding=1),
    nn.BatchNorm2d(128),
    nn.ReLU(),
    nn.Conv2d(128, 128, 3, padding=1),
    nn.BatchNorm2d(128),
    nn.ReLU(),
    nn.Dropout(.2),
    nn.MaxPool2d(2),

    nn.Conv2d(128, 256, 3, padding=1),
    nn.BatchNorm2d(256),
    nn.ReLU(),
    nn.Conv2d(256, 256, 3, padding=1),
    nn.BatchNorm2d(256),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Flatten(),
    nn.Linear(256, 1024),
    nn.Dropout(.3),
    nn.ReLU(),
    nn.Linear(1024, 10)
)

model_cnn = model_cnn.to(device)

In [8]:
epochs = 20
loss_function = nn.CrossEntropyLoss()
optimizer = Adam(model_cnn.parameters())

cnn_res = train(model_cnn, train_loader_from_tv_at_ONLY_32_batch, loss_function, optimizer, epochs)

Training: 100%|██████████| 20/20 [02:45<00:00,  8.28s/epoch, train_acc=0.9654, train_loss=0.1012]


In [11]:
cnn_test_loss, cnn_test_acc = test(model_cnn, test_loader_from_tv_at_ONLY_32_batch, loss_function)

In [12]:
comparison_dict = {
    'Modelo': ['CNN'],
    'Pérdida de prueba': [cnn_test_loss],
    'Exactitud de prueba': [cnn_test_acc]
}

pd.DataFrame(comparison_dict)

,Modelo,Pérdida de prueba,Exactitud de prueba
0,CNN,1.103838,0.7621
